# GIAI ĐOẠN 2: LÀM SẠCH NHẸ & XỬ LÝ TRÙNG LẶP DỮ LIỆU (LIGHT PREPROCESSING)

## 1. Mục tiêu và Bản chất giai đoạn
- **Làm sạch nhẹ (Light Cleaning)**: Là bước chuẩn hóa văn bản ở mức độ bề mặt, loại bỏ các yếu tố rác kỹ thuật và che thông tin nhạy cảm nhưng **tuyệt đối KHÔNG làm biến đổi cấu trúc ngữ pháp hay ý nghĩa cảm xúc** của câu.
- **Các nhiệm vụ chính**:
  1. **Xóa dòng rỗng / khuyết thiếu (Empty & NaN removal)**: Loại bỏ các review không có nội dung chữ hoặc chỉ toàn dấu khoảng trắng.
  2. **Chuẩn hóa Unicode (NFC)**: Đưa các ký tự tiếng Việt về chuẩn dựng sẵn (tránh lỗi 2 câu nhìn giống nhau nhưng mã byte khác nhau).
  3. **Che thông tin cá nhân (PII Masking)**: Thay thế số điện thoại, email, link web thành các token đại diện như `[sđt]`, `[email]`, `[link]`.
  4. **Chuẩn hóa khoảng trắng**: Xóa bỏ các ký tự xuống dòng `\n`, tab `\t` và gộp nhiều dấu cách thừa thành 1 khoảng trắng duy nhất.
  5. **Lọc bỏ review vô nghĩa**: Loại bỏ các review chỉ toàn dấu chấm `...`, số hoặc icon; giữ lại các từ cảm xúc ngắn có nghĩa (như 'tốt', 'tệ', 'ok').
  6. **Khử trùng lặp (Deduplication)**: Chuẩn hóa nội dung so khớp (viết thường, bỏ dấu câu thừa) để gom các câu giống nhau về một bản ghi duy nhất, nhưng vẫn bảo toàn câu gốc tự nhiên cho bước gán nhãn.

- **Đầu vào**: `data/raw/reviews_....csv` (Dữ liệu cào từ Giai đoạn 1).
- **Đầu ra**: `data/processed/reviews_clean.csv`.

### Bước 1: Khai báo thư viện và đọc dữ liệu thô

In [ ]:
import re
import unicodedata
from pathlib import Path
import pandas as pd

# Đường dẫn file đầu vào và đầu ra
INPUT_FILE = Path("../data/raw/reviews_1000.csv")  # Có thể đổi thành reviews_50.csv hoặc file bất kỳ
OUTPUT_FILE = Path("../data/processed/reviews_clean.csv")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# Đọc dữ liệu thô
df_raw = pd.read_csv(INPUT_FILE)
print(f"-> Đã nạp dữ liệu từ: {INPUT_FILE.resolve()}")
print(f"-> Tổng số dòng ban đầu: {len(df_raw)} dòng.\n")

# Xem 3 dòng đầu tiên
df_raw.head(3)

### Bước 2: Xây dựng các hàm chuẩn hóa văn bản (Code tay rõ ràng từng quy tắc)

In [ ]:
# 1. Biểu thức chính quy (Regex) để nhận diện thông tin nhạy cảm
REGEX_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
REGEX_URL   = re.compile(r"https?://\S+|www\.\S+")
# Nhận diện số điện thoại Việt Nam (đầu 0 hoặc +84 kèm 9-10 chữ số, cho phép có khoảng trắng/chấm/gạch nối)
REGEX_PHONE = re.compile(r"(?<!\d)(?:\+?84|0)[1-9](?:[\s.\-]?\d){7,9}(?!\d)")

def clean_light(text: str) -> str:
    """
    Làm sạch nhẹ văn bản mà không làm mất ý nghĩa câu:
    1. Kiểm tra rỗng/NaN.
    2. Chuẩn hóa Unicode về NFC.
    3. Che thông tin liên hệ: SĐT -> [sđt], Email -> [email], Link -> [link].
    4. Gộp nhiều khoảng trắng/xuống dòng thừa thành một dấu cách.
    """
    if not isinstance(text, str) or not text.strip():
        return ""
        
    # 1. Chuẩn hóa bảng mã Unicode NFC
    text = unicodedata.normalize("NFC", text)
    
    # 2. Che thông tin cá nhân
    text = REGEX_EMAIL.sub("[email]", text)
    text = REGEX_URL.sub("[link]", text)
    text = REGEX_PHONE.sub("[sđt]", text)
    
    # 3. Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

def normalize_for_matching(text: str) -> str:
    """
    Chuẩn hóa riêng cho việc SO SÁNH TRÙNG LẶP (Deduplication):
    - Viết thường (.lower())
    - Bỏ toàn bộ dấu câu (, . ! ? ...)
    - Chuẩn hóa NFC và gộp khoảng trắng
    Giúp nhận diện chính xác các câu cùng nội dung dù khác chữ hoa hay dấu câu.
    """
    text = clean_light(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()

def is_valid_review(text: str) -> bool:
    """
    Kiểm tra xem review có chứa nội dung chữ hợp lệ hay không:
    - Không được rỗng (sau khi đã strip).
    - Phải có ít nhất 1 chữ cái tiếng Việt/Latinh (loại bỏ trường hợp chỉ toàn dấu chấm '...', số hoặc icon).
    - Cho phép review ngắn 1 từ có nghĩa (như 'tốt', 'tệ', 'ok'), loại bỏ ký tự đơn lẻ vô nghĩa (như 'a', '.').
    """
    if not text or not text.strip():
        return False
        
    has_alpha = bool(re.search(r"[a-zA-ZÀ-ỹ]", text))
    if not has_alpha:
        return False
        
    words = text.split()
    if len(words) == 1 and len(words[0]) < 2:
        return False
        
    return True

### Bước 3: Áp dụng lên toàn bộ tập dữ liệu (Xóa rỗng, lọc rác & Khử trùng lặp)

In [ ]:
df_clean = df_raw.copy()
n_initial = len(df_clean)

# 1. XÓA BỎ DÒNG RỖNG / NaN NGAY TỪ ĐẦU
df_clean.dropna(subset=["noi_dung"], inplace=True)
df_clean = df_clean[df_clean["noi_dung"].astype(str).str.strip() != ""].copy()
n_after_dropna = len(df_clean)
n_empty_removed = n_initial - n_after_dropna

# 2. Áp dụng hàm làm sạch nhẹ
df_clean["noi_dung_clean"] = df_clean["noi_dung"].apply(clean_light)

# 3. Lọc bỏ các dòng không hợp lệ (không có chữ, chỉ toàn số/icon/ký tự rác)
valid_mask = df_clean["noi_dung_clean"].apply(is_valid_review)
df_clean = df_clean[valid_mask].copy()
n_after_valid = len(df_clean)
n_invalid_removed = n_after_dropna - n_after_valid

# 4. Khử trùng lặp nội dung (Deduplication) dựa trên khóa chuẩn hóa (viết thường, bỏ dấu câu)
df_clean["noi_dung_so_trung"] = df_clean["noi_dung_clean"].apply(normalize_for_matching)
df_clean = df_clean.drop_duplicates(subset=["noi_dung_so_trung"]).copy()
df_clean.drop(columns=["noi_dung_so_trung"], inplace=True)

n_after_dedup = len(df_clean)
n_duplicates_removed = n_after_valid - n_after_dedup

# Gán lại nội dung sạch vào cột chính
df_clean["noi_dung"] = df_clean["noi_dung_clean"]
df_clean.drop(columns=["noi_dung_clean"], inplace=True)

# In bảng tổng kết quá trình làm sạch nhẹ rõ ràng từng chỉ số
print("="*55)
print("TỔNG KẾT QUÁ TRÌNH LÀM SẠCH NHẸ (LIGHT PREPROCESSING):")
print(f"- Số lượng dòng ban đầu       : {n_initial}")
print(f"- Loại bỏ do review rỗng / NaN: -{n_empty_removed}")
print(f"- Loại bỏ do câu rác / vô nghĩa: -{n_invalid_removed}")
print(f"- Loại bỏ do trùng lặp nội dung: -{n_duplicates_removed}")
print(f"- Số lượng dòng sạch còn lại   : {n_after_dedup} (Giữ lại {n_after_dedup/n_initial*100:.1f}%)")
print("="*55)

### Bước 4: Lưu dữ liệu đã làm sạch vào `data/processed/`

In [ ]:
# Lưu file dữ liệu sạch
df_clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"[THÀNH CÔNG] Dữ liệu sạch đã được lưu tại: {OUTPUT_FILE.resolve()}\n")

# Kiểm tra lại phân bố cảm xúc sau khi lọc
print("Phân bố cảm xúc sau khi làm sạch nhẹ:")
print(df_clean["nhom_cam_xuc"].value_counts())

# Hiển thị 5 dòng ngẫu nhiên để kiểm tra
df_clean.sample(min(5, len(df_clean)))